In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

# =========================================================
# STEP 1: Load Image & Model
# =========================================================
# Load your image
image_path = "/home/prashanna/Documents/PROJECT VBMD/pic.jpeg"  # Replace with your image filename
image = Image.open(image_path).convert("RGB")

# DPT-Large or GLPN yield metric depth estimates
model_name = "intel/dpt-large"  # High quality depth transformer

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForDepthEstimation.from_pretrained(model_name)

# =========================================================
# STEP 2: Inference & Convert Inverse Depth to Actual Depth
# =========================================================
# Prepare tensor inputs
inputs = processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    predicted_depth = outputs.predicted_depth

# Interpolate back to original input image resolution (H, W)
prediction = torch.nn.functional.interpolate(
    predicted_depth.unsqueeze(1),
    size=image.size[::-1],  # PIL size is (W, H), PyTorch needs (H, W)
    mode="bicubic",
    align_corners=False,
).squeeze()

# Convert tensor to numpy array
raw_output = prediction.numpy()

# Determine if the model output is raw depth or disparity.
# For disparity-based models: Depth = 1 / (Disparity + epsilon)
# Add a small epsilon (1e-6) to prevent division by zero errors
true_depth_meters = 1.0 / (raw_output + 1e-6)

# Clip extreme outlier distances (e.g., sky or reflection artifacts) for clean plotting
min_dist = np.percentile(true_depth_meters, 1)   # 1st percentile
max_dist = np.percentile(true_depth_meters, 99)  # 99th percentile
clipped_depth = np.clip(true_depth_meters, min_dist, max_dist)

# =========================================================
# STEP 3: Display RGB & Actual Metric Depth Map
# =========================================================
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: RGB Image
ax[0].imshow(image)
ax[0].set_title("Original RGB Image")
ax[0].axis("off")

# Plot 2: True Metric Depth (Dark = Near, Bright/Yellow = Far)
# Using 'magma' colormap where darker colors = small meters (close) and lighter colors = large meters (far)
depth_plot = ax[1].imshow(clipped_depth, cmap="magma")
ax[1].set_title("Actual Depth Map (Meters)")
ax[1].axis("off")

# Add metric colorbar
cbar = fig.colorbar(depth_plot, ax=ax[1], shrink=0.75)
cbar.set_label("Estimated Distance (Meters)", rotation=270, labelpad=15)

plt.tight_layout()
plt.show()

# =========================================================
# STEP 4: Inspect Distance in Meters at Specific Pixels
# =========================================================
def get_pixel_depth(x, y, depth_matrix):
    """
    Query depth value in meters at coordinate (x, y).
    x = column (width axis)
    y = row (height axis)
    """
    distance = depth_matrix[y, x]
    print(f"📍 Distance at Pixel (x={x}, y={y}): {distance:.2f} meters")
    return distance

# Example inspection points
h, w = true_depth_meters.shape
print("\n--- Pixel Distance Values ---")
get_pixel_depth(w // 2, h // 2, true_depth_meters)       # Center pixel
get_pixel_depth(w // 2, int(h * 0.85), true_depth_meters) # Foreground / Bottom area
get_pixel_depth(w // 2, int(h * 0.2), true_depth_meters)  # Background / Top area